In [0]:
# Load the table into a Spark DataFrame
df = spark.table("user_behaviour_catalog.user_behaviour_schema.cust_behavior_2")

# Display the first few rows
display(df)


In [0]:
# Print schema
df.printSchema()

# Summary statistics
df.describe().show()


In [0]:
%python
# Count of nulls per column
from pyspark.sql.functions import col, isnan, when, count

df.select([
    count(when(col(c).isNull() | (col(c).cast("string") == "") if c != "FLOAT" and c != "DOUBLE" else isnan(c), c)).alias(c)
    for c in df.columns
]).show()

# Distinct values in key categorical columns
df.select("Membership Type", "City", "Satisfaction Level").distinct().show()

In [0]:
%python
from pyspark.sql.functions import col, when
import pandas as pd

# Step 1: Define churn based on custom conditions
df = df.withColumn(
    'churn',
    when(
        (col('Days Since Last Purchase') > 15) &
        (col('Satisfaction Level') == 'Unsatisfied') &
        (col('Average Rating') < 4),
        1
    ).otherwise(0)
)

# Step 2: Prepare data for linear regression
# Drop rows with missing values in selected columns
features = ['Age', 'Items Purchased', 'Average Rating', 'Discount Applied', 'Total Spend']
df_model = df.select(features + ['churn']).dropna()

# Convert Spark DataFrame to Pandas DataFrame for scikit-learn
df_model_pd = df_model.toPandas()

# Step 3: Build a Linear Regression model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = df_model_pd[features]
y = df_model_pd['churn']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict and evaluate
y_pred = lr_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})



display(coefficients)
display(mse)
display(r2)

In [0]:
from sklearn.metrics import precision_score, recall_score

# Convert continuous predictions to binary using a 0.5 threshold
y_pred_binary = (y_pred >= 0.5).astype(int)

# Calculate precision and recall
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)

precision, recall


In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on test set
y_pred = log_reg.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

accuracy, precision, recall
